# Estudo Geoespacial: Consumo de Combustivel vs Declividade dos Talhoes

Objetivo: unir telemetria georreferenciada, raster de declividade e limites de talhao para responder:

1. Como o consumo varia com a declividade?
2. Quais talhoes e maquinas concentram maior intensidade de consumo?
3. Onde ficam os eventos de sem apontamento e paradas especificas?
4. Quais oportunidades de viabilidade e retorno (R$) aparecem no periodo?


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 260)


In [ ]:
# Parametros principais
DATA_PATH = Path('telemetria_colheita_extraida/consolidado_telemetria_colheita.csv')
RASTER_DECLIV_PATH = Path('declividade_area_talhoes.tif')
FIELDS_PATH = Path('fields.mbtiles')

PRECO_DIESEL_R_L = 6.50
PARADA_ALVO = 'FALTA CAMINHAO'
MIN_HORAS_TALHAO = 5

assert DATA_PATH.exists(), f'Arquivo nao encontrado: {DATA_PATH}'
assert RASTER_DECLIV_PATH.exists(), f'Arquivo nao encontrado: {RASTER_DECLIV_PATH}'
assert FIELDS_PATH.exists(), f'Arquivo nao encontrado: {FIELDS_PATH}'


In [ ]:
# 1) Carga e limpeza base

df = pd.read_csv(DATA_PATH, sep=';', low_memory=False)

num_cols = [
    'vl_tempo_segundos', 'vl_consumo_instantaneo', 'vl_hectares_hora',
    'vl_velocidade', 'vl_rpm', 'latitude_interpolacao', 'longitude_interpolacao'
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['dt_hr_local_inicial'] = pd.to_datetime(df['dt_hr_local_inicial'], errors='coerce')
df = df.dropna(subset=['dt_hr_local_inicial']).copy()

df['cd_equipamento'] = df['cd_equipamento'].astype(str)
df['cd_estado'] = df['cd_estado'].astype(str).str.upper().str.strip()
df['desc_parada'] = df['desc_parada'].fillna('*NAO_DEFINIDO*').astype(str).str.strip()
df['desc_operador'] = df['desc_operador'].fillna('*NAO_DEFINIDO*').astype(str).str.strip()

estado_map = {
    'E': 'Efetivo',
    'P': 'Parada',
    'F': 'Parada',
    'M': 'Manobra',
    'D': 'Deslocamento',
}
df['estado_processo'] = df['cd_estado'].map(estado_map).fillna('Outros')

df['data'] = df['dt_hr_local_inicial'].dt.date
df['hora'] = df['dt_hr_local_inicial'].dt.hour

valid_coord = df['latitude_interpolacao'].between(-90, 90) & df['longitude_interpolacao'].between(-180, 180)
valid_oper = (df['vl_tempo_segundos'].fillna(0) > 0) & (df['vl_consumo_instantaneo'].fillna(0) >= 0)

df_base = df[valid_coord & valid_oper].copy()
df_base['horas_evento'] = df_base['vl_tempo_segundos'] / 3600.0
# Premissa: vl_consumo_instantaneo em L/h
df_base['consumo_l_estimado'] = df_base['vl_consumo_instantaneo'].clip(lower=0) * df_base['horas_evento']

print('Linhas telemetria bruta:', len(df))
print('Linhas validas georreferenciadas:', len(df_base))
print('Periodo:', df_base['dt_hr_local_inicial'].min(), '->', df_base['dt_hr_local_inicial'].max())


In [ ]:
# 2) Enriquecimento espacial: declividade por ponto + talhao por spatial join

# 2.1 Sample do raster de declividade
with rasterio.open(RASTER_DECLIV_PATH) as src:
    decliv = src.read(1).astype('float64')
    if src.nodata is not None:
        decliv[decliv == src.nodata] = np.nan
    inv = ~src.transform

    cols_f, rows_f = inv * (
        df_base['longitude_interpolacao'].to_numpy(),
        df_base['latitude_interpolacao'].to_numpy()
    )
    rows = np.floor(rows_f).astype(int)
    cols = np.floor(cols_f).astype(int)
    inside = (rows >= 0) & (rows < decliv.shape[0]) & (cols >= 0) & (cols < decliv.shape[1])

    slope_values = np.full(len(df_base), np.nan)
    slope_values[inside] = decliv[rows[inside], cols[inside]]


df_base['declividade_pct'] = slope_values

# filtros de qualidade
df_base = df_base[df_base['declividade_pct'].notna()].copy()
df_base = df_base[(df_base['declividade_pct'] >= 0) & (df_base['declividade_pct'] <= 80)].copy()
df_base = df_base[df_base['vl_consumo_instantaneo'] <= 120].copy()

# 2.2 Join com talhoes
geo_pts = gpd.GeoDataFrame(
    df_base.copy(),
    geometry=gpd.points_from_xy(df_base['longitude_interpolacao'], df_base['latitude_interpolacao']),
    crs='EPSG:4326'
)

talhoes = gpd.read_file(FIELDS_PATH)
if talhoes.crs is None:
    talhoes = talhoes.set_crs('EPSG:4326')
elif str(talhoes.crs) != 'EPSG:4326':
    talhoes = talhoes.to_crs('EPSG:4326')

join_cols = [c for c in ['DESC_TALHA', 'TALHAO', 'NOME_FAZ', 'FAZENDA', 'ZONA', 'AREA_TOTAL'] if c in talhoes.columns]
geo = gpd.sjoin(geo_pts, talhoes[join_cols + ['geometry']], how='left', predicate='within')
geo['talhao_label'] = geo['DESC_TALHA'].fillna('SEM_TALHAO').astype(str)

print('Linhas apos enriquecimento espacial:', len(geo))
print('Talhoes identificados:', geo['talhao_label'].nunique())


## 3) Consumo vs declividade (visao macro)


In [ ]:
ana = geo[geo['estado_processo'].isin(['Efetivo', 'Deslocamento', 'Manobra'])].copy()
ana = ana[ana['horas_evento'] > 0].copy()

bins = [0, 3, 8, 20, 45, np.inf]
labels = ['0-3%', '3-8%', '8-20%', '20-45%', '>45%']
ana['classe_decliv'] = pd.cut(ana['declividade_pct'], bins=bins, labels=labels, right=False).astype(str)
ana = ana[ana['classe_decliv'] != 'nan'].copy()

cons_decliv = (
    ana.groupby('classe_decliv', as_index=False)
       .agg(horas=('horas_evento', 'sum'), litros=('consumo_l_estimado', 'sum'))
)
order = pd.Categorical(cons_decliv['classe_decliv'], categories=labels, ordered=True)
cons_decliv = cons_decliv.assign(_ord=order).sort_values('_ord').drop(columns='_ord')
cons_decliv['l_h'] = cons_decliv['litros'] / cons_decliv['horas']

baseline = cons_decliv.loc[cons_decliv['classe_decliv'].eq('0-3%'), 'l_h']
baseline = baseline.iloc[0] if len(baseline) else np.nan
cons_decliv['delta_l_h_vs_0_3'] = cons_decliv['l_h'] - baseline
cons_decliv['delta_pct_vs_0_3'] = (cons_decliv['l_h'] / baseline - 1) * 100 if pd.notna(baseline) and baseline > 0 else np.nan

cons_decliv


In [ ]:
fig = px.bar(
    cons_decliv,
    x='classe_decliv',
    y='l_h',
    text='l_h',
    title='Consumo medio (L/h) por faixa de declividade',
    labels={'classe_decliv': 'Faixa de declividade', 'l_h': 'Consumo medio (L/h)'}
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside', marker_color='#1d4e89')
fig.update_layout(template='plotly_white', height=420)
fig.show()


## 4) Mapa georreferenciado dentro dos talhoes


In [ ]:
# Amostragem para manter mapa responsivo
map_df = ana[['latitude_interpolacao', 'longitude_interpolacao', 'cd_equipamento', 'talhao_label', 'declividade_pct', 'vl_consumo_instantaneo', 'estado_processo']].dropna().copy()
if len(map_df) > 50000:
    map_df = map_df.sample(50000, random_state=42)

fig = px.scatter_mapbox(
    map_df,
    lat='latitude_interpolacao',
    lon='longitude_interpolacao',
    color='vl_consumo_instantaneo',
    color_continuous_scale='Turbo',
    hover_data=['cd_equipamento', 'talhao_label', 'declividade_pct', 'estado_processo'],
    zoom=10,
    height=620,
    title='Pontos georreferenciados: intensidade de consumo (L/h)'
)
fig.update_layout(mapbox_style='carto-positron', template='plotly_white', margin=dict(l=10, r=10, t=40, b=10))
fig.show()


In [ ]:
# Choropleth por talhao: consumo medio L/h e declividade media
by_talhao = (
    ana.groupby('talhao_label', as_index=False)
       .agg(
           horas=('horas_evento', 'sum'),
           litros=('consumo_l_estimado', 'sum'),
           decliv_media=('declividade_pct', 'mean')
       )
)
by_talhao = by_talhao[by_talhao['horas'] >= MIN_HORAS_TALHAO].copy()
by_talhao['l_h'] = by_talhao['litros'] / by_talhao['horas']

map_talhoes = talhoes.copy()
if 'DESC_TALHA' in map_talhoes.columns:
    map_talhoes['talhao_label'] = map_talhoes['DESC_TALHA'].astype(str)
else:
    map_talhoes['talhao_label'] = map_talhoes.index.astype(str)

map_talhoes = map_talhoes.merge(by_talhao, on='talhao_label', how='left')
map_talhoes[['talhao_label', 'horas', 'l_h', 'decliv_media']].sort_values('l_h', ascending=False).head(15)


In [ ]:
if map_talhoes['l_h'].notna().any():
    fig = px.choropleth_mapbox(
        map_talhoes,
        geojson=map_talhoes.geometry,
        locations=map_talhoes.index,
        color='l_h',
        color_continuous_scale='YlOrRd',
        hover_name='talhao_label',
        hover_data={'horas':':.1f', 'decliv_media':':.2f', 'l_h':':.2f'},
        center={'lat': float(map_df['latitude_interpolacao'].median()), 'lon': float(map_df['longitude_interpolacao'].median())},
        zoom=10,
        opacity=0.65,
        title='Talhoes: intensidade media de consumo (L/h)',
        height=620
    )
    fig.update_layout(mapbox_style='carto-positron', template='plotly_white', margin=dict(l=10, r=10, t=40, b=10))
    fig.show()


## 5) Maquinas x declividade (benchmark operacional)


In [ ]:
mxs = (
    ana.groupby(['cd_equipamento', 'classe_decliv'], as_index=False)
       .agg(horas=('horas_evento', 'sum'), litros=('consumo_l_estimado', 'sum'))
)
mxs = mxs[mxs['horas'] >= 2].copy()
mxs['l_h'] = mxs['litros'] / mxs['horas']
mxs.head(20)


In [ ]:
fig = px.bar(
    mxs,
    x='cd_equipamento',
    y='l_h',
    color='classe_decliv',
    barmode='group',
    title='Consumo por maquina dentro de cada classe de declividade (L/h)',
    labels={'cd_equipamento': 'Maquina', 'l_h': 'Consumo (L/h)', 'classe_decliv': 'Declividade'}
)
fig.update_layout(template='plotly_white', height=500)
fig.show()


## 6) Sem apontamento e picos de parada especifica


In [ ]:
sem_ap = (
    geo['desc_parada'].str.upper().eq('*NAO_DEFINIDO*')
    | geo['desc_operador'].str.upper().eq('*NAO_DEFINIDO*')
    | geo['cd_operador'].astype(str).str.strip().eq('-1')
)
geo_sem = geo[sem_ap].copy()

sem_por_talhao = (
    geo_sem.groupby('talhao_label', as_index=False)
           .agg(horas_sem=('horas_evento', 'sum'), eventos=('talhao_label', 'size'))
           .sort_values('horas_sem', ascending=False)
)
sem_por_talhao.head(15)


In [ ]:
if not geo_sem.empty:
    sem_map = geo_sem[['latitude_interpolacao', 'longitude_interpolacao', 'horas_evento', 'talhao_label']].dropna().copy()
    if len(sem_map) > 50000:
        sem_map = sem_map.sample(50000, random_state=42)

    fig = px.density_mapbox(
        sem_map,
        lat='latitude_interpolacao',
        lon='longitude_interpolacao',
        z='horas_evento',
        radius=12,
        zoom=10,
        mapbox_style='carto-positron',
        height=620,
        title='Concentracao espacial de sem apontamento (ponderado por horas)'
    )
    fig.update_layout(template='plotly_white', margin=dict(l=10, r=10, t=40, b=10))
    fig.show()


In [ ]:
geo_par = geo[geo['estado_processo'].eq('Parada')].copy()
geo_par['parada_norm'] = geo_par['desc_parada'].str.upper().str.strip()

alvo = geo_par[geo_par['parada_norm'].str.contains(PARADA_ALVO.upper().strip(), na=False)].copy()

pico_hora = (
    alvo.groupby('hora', as_index=False)
        .agg(horas=('horas_evento', 'sum'), eventos=('hora', 'size'))
        .sort_values('hora')
)

pico_hora


In [ ]:
if not pico_hora.empty:
    fig = go.Figure()
    fig.add_bar(x=pico_hora['hora'], y=pico_hora['horas'], name='Horas', marker_color='#c1121f')
    fig.add_trace(go.Scatter(x=pico_hora['hora'], y=pico_hora['eventos'], mode='lines+markers', name='Eventos', yaxis='y2', line=dict(color='#1d4e89')))
    fig.update_layout(
        template='plotly_white',
        height=430,
        title=f'Picos horarios da parada: {PARADA_ALVO}',
        xaxis_title='Hora do dia',
        yaxis=dict(title='Horas de parada'),
        yaxis2=dict(title='Qtd de eventos', overlaying='y', side='right')
    )
    fig.show()


## 7) Viabilidade e retorno (cenarios)

Cenario A (eficiencia em parada):
- aplica reducao de 10% e 20% no consumo em eventos de parada com consumo > 0.

Cenario B (declividade):
- compara consumo por classe de declividade; identifica diferenca relativa vs faixa 0-3%.
- observacao importante: se houver poucas horas em classes altas, o retorno potencial fica limitado.


In [ ]:
# Cenario A
par = geo[geo['estado_processo'].eq('Parada')].copy()
par_sum = (
    par.groupby('desc_parada', as_index=False)
       .agg(horas=('horas_evento', 'sum'), litros=('consumo_l_estimado', 'sum'))
)
par_mean = par.groupby('desc_parada', as_index=False)['vl_consumo_instantaneo'].mean().rename(columns={'vl_consumo_instantaneo': 'l_h_medio'})
par_sum = par_sum.merge(par_mean, on='desc_parada', how='left')

par_sum['litros_pot_10'] = np.where(par_sum['l_h_medio'] > 0, par_sum['litros'] * 0.10, 0)
par_sum['litros_pot_20'] = np.where(par_sum['l_h_medio'] > 0, par_sum['litros'] * 0.20, 0)

pot10_l = par_sum['litros_pot_10'].sum()
pot20_l = par_sum['litros_pot_20'].sum()

ret10 = pot10_l * PRECO_DIESEL_R_L
ret20 = pot20_l * PRECO_DIESEL_R_L

res_viab = pd.DataFrame([
    {'cenario': 'Reducao 10% consumo em parada', 'litros_potenciais': pot10_l, 'retorno_r$': ret10},
    {'cenario': 'Reducao 20% consumo em parada', 'litros_potenciais': pot20_l, 'retorno_r$': ret20},
])

res_viab


In [ ]:
# Cenario B (declividade)
cons_decliv[['classe_decliv', 'horas', 'l_h', 'delta_l_h_vs_0_3', 'delta_pct_vs_0_3']]


In [ ]:
# 8) Exportacao de resultados
outdir = Path('analises_resultados')
outdir.mkdir(exist_ok=True)

cons_decliv.to_csv(outdir / 'consumo_declividade_resumo.csv', sep=';', index=False)
by_talhao.to_csv(outdir / 'consumo_talhao_ranking.csv', sep=';', index=False)
mxs.to_csv(outdir / 'consumo_maquina_classe_declividade.csv', sep=';', index=False)
sem_por_talhao.to_csv(outdir / 'sem_apontamento_talhao.csv', sep=';', index=False)
if not pico_hora.empty:
    pico_hora.to_csv(outdir / 'picos_horarios_parada_alvo.csv', sep=';', index=False)
res_viab.to_csv(outdir / 'cenarios_viabilidade_combustivel.csv', sep=';', index=False)

# camada geoespacial para SIG/QGIS
geo_export_cols = [
    'dt_hr_local_inicial', 'data', 'hora', 'cd_equipamento', 'estado_processo',
    'desc_parada', 'talhao_label', 'declividade_pct', 'horas_evento',
    'vl_consumo_instantaneo', 'consumo_l_estimado', 'geometry'
]
geo[geo_export_cols].to_file(outdir / 'eventos_operacionais_georef.gpkg', layer='eventos', driver='GPKG')

print('Arquivos exportados em:', outdir.resolve())


## Leitura executiva rapida

1. Se `delta_pct_vs_0_3` subir nas classes altas, ha sensibilidade do consumo a declividade.
2. Talhoes com `l_h` alto + `decliv_media` alta entram como prioridade de rota/parametrizacao.
3. `sem_por_talhao` orienta acao de qualidade de apontamento por area.
4. Cenarios de viabilidade mostram retorno financeiro direto com reducao de consumo em parada.


## 8) Direcao das operacoes (rumo) e impacto no consumo

Objetivo: verificar se existe diferenca de eficiencia por direcao de deslocamento/opera??o e estimar ganho potencial com ajuste de orientacao.

Regras de qualidade aplicadas para evitar erro de interpretacao:
- coordenadas iniciais/finais validas
- duracao > 0 e <= 2h por evento
- consumo instantaneo dentro de faixa plausivel (0 a 120 L/h)
- distancia minima por segmento (>= 5 m)
- velocidade implicita maxima de 72 km/h
- estados de interesse: Efetivo, Deslocamento, Manobra

In [ ]:
# Calculo de rumo (bearing) com QA forte

dir_df = geo.copy()

for c in ['vl_latitude_inicial', 'vl_longitude_inicial', 'vl_latitude_final', 'vl_longitude_final']:
    if c in dir_df.columns:
        dir_df[c] = pd.to_numeric(dir_df[c], errors='coerce')

coord_ok = (
    dir_df['vl_latitude_inicial'].between(-90, 90)
    & dir_df['vl_latitude_final'].between(-90, 90)
    & dir_df['vl_longitude_inicial'].between(-180, 180)
    & dir_df['vl_longitude_final'].between(-180, 180)
)

dur_ok = dir_df['vl_tempo_segundos'].fillna(0).between(1, 7200)
cons_ok = dir_df['vl_consumo_instantaneo'].fillna(0).between(0, 120)
state_ok = dir_df['estado_processo'].isin(['Efetivo', 'Deslocamento', 'Manobra'])

dir_df = dir_df[coord_ok & dur_ok & cons_ok & state_ok].copy()

lat1 = np.deg2rad(dir_df['vl_latitude_inicial'])
lat2 = np.deg2rad(dir_df['vl_latitude_final'])
dlon = np.deg2rad(dir_df['vl_longitude_final'] - dir_df['vl_longitude_inicial'])

y = np.sin(dlon) * np.cos(lat2)
x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
dir_df['bearing_deg'] = (np.degrees(np.arctan2(y, x)) + 360) % 360

# Distancia haversine para valida??o
R = 6371000.0
dphi = np.deg2rad(dir_df['vl_latitude_final'] - dir_df['vl_latitude_inicial'])
dlam = np.deg2rad(dir_df['vl_longitude_final'] - dir_df['vl_longitude_inicial'])
a = np.sin(dphi / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlam / 2.0) ** 2
c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
dir_df['dist_m'] = R * c
dir_df['speed_m_s_impl'] = dir_df['dist_m'] / dir_df['vl_tempo_segundos']

move_ok = (dir_df['dist_m'] >= 5) & (dir_df['speed_m_s_impl'] <= 20)
dir_df = dir_df[move_ok].copy()

dir_df['horas_evento'] = dir_df['vl_tempo_segundos'] / 3600.0
dir_df['consumo_l_estimado'] = dir_df['vl_consumo_instantaneo'] * dir_df['horas_evento']

# Direcoes em 8 setores
labels_dir = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
sector = ((dir_df['bearing_deg'] + 22.5) // 45) % 8
dir_df['direcao_setor'] = sector.map({i: l for i, l in enumerate(labels_dir)})

print('Registros validos para analise de direcao:', len(dir_df))
print('Periodo:', dir_df['dt_hr_local_inicial'].min(), '->', dir_df['dt_hr_local_inicial'].max())


In [ ]:
# Eficiencia por direcao

dir_sum = (
    dir_df.groupby('direcao_setor', as_index=False)
          .agg(
              horas=('horas_evento', 'sum'),
              litros=('consumo_l_estimado', 'sum'),
              dist_m=('dist_m', 'sum'),
              hectares=('vl_hectares_hora', lambda s: (s.fillna(0) * dir_df.loc[s.index, 'horas_evento']).sum())
          )
)

dir_sum['dist_km'] = dir_sum['dist_m'] / 1000.0
dir_sum['l_h'] = dir_sum['litros'] / dir_sum['horas']
dir_sum['l_km'] = dir_sum['litros'] / dir_sum['dist_km']
dir_sum['l_ha'] = np.where(dir_sum['hectares'] > 0, dir_sum['litros'] / dir_sum['hectares'], np.nan)

ord = pd.Categorical(dir_sum['direcao_setor'], categories=['N','NE','E','SE','S','SW','W','NW'], ordered=True)
dir_sum = dir_sum.assign(_ord=ord).sort_values('_ord').drop(columns='_ord')

dir_sum


In [ ]:
fig = px.bar(
    dir_sum,
    x='direcao_setor',
    y='l_h',
    text='l_h',
    title='Consumo medio por direcao de operacao (L/h)',
    labels={'direcao_setor': 'Direcao', 'l_h': 'Consumo medio (L/h)'}
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside', marker_color='#1d4e89')
fig.update_layout(template='plotly_white', height=420)
fig.show()

fig2 = px.bar(
    dir_sum,
    x='direcao_setor',
    y='l_km',
    text='l_km',
    title='Litros por km por direcao (eficiencia de deslocamento)',
    labels={'direcao_setor': 'Direcao', 'l_km': 'L/km'}
)
fig2.update_traces(texttemplate='%{text:.2f}', textposition='outside', marker_color='#2a9d8f')
fig2.update_layout(template='plotly_white', height=420)
fig2.show()


In [ ]:
# Mapa de rumo (amostrado)
map_dir = dir_df[['latitude_interpolacao','longitude_interpolacao','direcao_setor','vl_consumo_instantaneo','talhao_label']].dropna().copy()
if len(map_dir) > 40000:
    map_dir = map_dir.sample(40000, random_state=42)

fig = px.scatter_mapbox(
    map_dir,
    lat='latitude_interpolacao',
    lon='longitude_interpolacao',
    color='direcao_setor',
    hover_data=['talhao_label','vl_consumo_instantaneo'],
    zoom=10,
    height=620,
    title='Distribuicao espacial das direcoes operacionais (setores)'
)
fig.update_layout(mapbox_style='carto-positron', template='plotly_white', margin=dict(l=10, r=10, t=40, b=10))
fig.show()


In [ ]:
# Cenario de ganho por orientacao
# Benchmark conservador: quartil de melhor consumo (P25) em L/h entre direcoes com base suficiente.

dir_base = dir_sum[dir_sum['horas'] >= 10].copy()
bench_l_h = dir_base['l_h'].quantile(0.25) if not dir_base.empty else np.nan

dir_gain = dir_base.copy()
dir_gain['delta_l_h_vs_bench'] = (dir_gain['l_h'] - bench_l_h).clip(lower=0)
dir_gain['litros_potenciais'] = dir_gain['delta_l_h_vs_bench'] * dir_gain['horas']
dir_gain['retorno_r$'] = dir_gain['litros_potenciais'] * PRECO_DIESEL_R_L

print('Benchmark direcional (P25 L/h):', round(float(bench_l_h), 3) if pd.notna(bench_l_h) else 'n/d')
print('Potencial total (L):', round(dir_gain['litros_potenciais'].sum(), 2))
print('Potencial total (R$):', round(dir_gain['retorno_r$'].sum(), 2))

dir_gain.sort_values('retorno_r$', ascending=False)


In [ ]:
# Exportacao complementar da analise de direcao
outdir = Path('analises_resultados')
outdir.mkdir(exist_ok=True)

dir_sum.to_csv(outdir / 'consumo_por_direcao_setor.csv', sep=';', index=False)
dir_gain.to_csv(outdir / 'cenario_ganho_por_direcao.csv', sep=';', index=False)

# Camada geoespacial com rumo para QGIS
geo_dir_cols = [
    'dt_hr_local_inicial', 'cd_equipamento', 'estado_processo', 'talhao_label',
    'bearing_deg', 'direcao_setor', 'dist_m', 'speed_m_s_impl',
    'vl_consumo_instantaneo', 'consumo_l_estimado', 'geometry'
]
dir_df[geo_dir_cols].to_file(outdir / 'eventos_com_direcao.gpkg', layer='direcao', driver='GPKG')

print('Exportados: consumo_por_direcao_setor.csv, cenario_ganho_por_direcao.csv, eventos_com_direcao.gpkg')


## 9) Check-list de confiabilidade antes de decidir

Antes de qualquer a??o de rota/dire??o:
1. Validar se houve troca de implemento/configura??o entre os dias comparados.
2. Separar por operador e janela clim?tica para evitar vi?s de contexto.
3. Exigir m?nimo de horas por dire??o/talh?o (ex.: >= 10h) para decis?o robusta.
4. Conferir se ganhos direcionais persistem quando filtramos apenas estado `Efetivo`.
5. Testar estabilidade semanal (n?o s? no agregado do per?odo).